# Fit the full microscope from Gaussian images

TemGym generates images by summing **complex Gaussian fields**, including interference. We measure those images, then fit a separate microscope through its transfer matrices. The fitter starts from guesses and does not receive the generating lens or source parameters.

In [ ]:
from pathlib import Path
import jax
from temgym_core import calibration

%matplotlib inline
jax.config.update("jax_enable_x64", True)

# Run from the repository root or this notebook's directory.
repo = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / "pyproject.toml").is_file())
experiment_directory = repo / "examples/generated/full_microscope_gaussian_pipeline"
print("JAX device:", jax.devices()[0])


The experiment uses **200 and 195 kV**, with objective current fixed. It records lens sweeps, calibrated specimen tilts and shifts, and currents through centred **4 and 10 µm apertures**. Apertures are retracted for imaging.

For this synthetic test, the specimen's amplitude and phase transmission and the detector response are known. We fit lens positions, lengths and strengths, separate source states at each voltage, and aperture position and centres. The lens model is coaxial.

The projector calibration assumes a known circular illumination profile, with unknown curvature and angular spread.

In [ ]:
# Supply acquisition settings. Output: saved images, currents and known commands.
experiment = calibration.simulate_experiment(
    output_directory=experiment_directory,
    voltages_v=(200_000., 195_000.),
    aperture_diameters_um=(4., 10.),
    centering_sigma_um=0.2,
    reuse_completed=True,
)
calibration.show_images(experiment_directory=experiment);

The ray order is $(x,y,\theta_x,\theta_y,1)^T$. TemGym builds

$$
T=\begin{pmatrix}
A_{xx}&A_{xy}&B_{xx}&B_{xy}&t_x\\
A_{yx}&A_{yy}&B_{yx}&B_{yy}&t_y\\
C_{xx}&C_{xy}&D_{xx}&D_{xy}&\eta_x\\
C_{yx}&C_{yy}&D_{yx}&D_{yy}&\eta_y\\
0&0&0&0&1
\end{pmatrix}.
$$

The shared lens positions, lengths and strengths determine these entries. Known tilts probe $B$; image changes and known shifts constrain the remaining parameters.

**First fit the projector.** Measure image centroids and shapes, with their uncertainties. The matrix model predicts these same quantities. With $R=(A\;B)$, exit-wave mean $m$ and covariance $S$,

$$m_d=Rm+t,\qquad S_d=RSR^T+S_{\mathrm{detector}}.$$

The predictor includes the known specimen's phase, diffraction and finite detector pixels. JAX differentiates these measurements with respect to lens parameters; it does not render pixels during fitting.
Independent starting guesses and engineering bounds are supplied by the editable [preset files](../../src/temgym_core/calibration/presets). The fitting calls accept replacement files.


In [ ]:
# Supply recorded projector images. Output: image measurements and covariance.
projector_measurements = calibration.measure_projector(experiment_directory=experiment)

# Supply measurements, independent guesses and engineering bounds.
# Output: shared projector geometry, uncertainty and results from four starts.
projector = calibration.fit_projector(
    measurements_directory=projector_measurements,
    number_of_starts=4,
)

**Then measure the condenser beam.** Three projector views estimate its position, angle, size, curvature and angular spread at each condenser setting. This step includes the fitted projector's uncertainty and corrects for finite detector pixels.

In [ ]:
# Supply condenser images and the calibrated projector.
# Output: independent specimen-beam measurements, with shared uncertainty.
beam_measurements = calibration.measure_condenser(
    experiment_directory=experiment,
    projector_fit=projector,
)

**Fit the complete microscope.** Beam estimates initialize the condenser fit. A final joint fit compares the original image measurements and aperture currents with both fitted halves, retaining their shared uncertainty.

We fit CL1, CL3, the mini lens, objective pre-field, sources and aperture geometry. The constraints are engineering bounds and the stated aperture-centering accuracy. The fitter also checks for alternative aperture centres.

In [ ]:
# Supply the saved image/beam/current measurements and separate condenser starting guesses.
# Output: both fitted halves and their joint covariance.
microscope = calibration.fit_microscope(
    beam_measurements_directory=beam_measurements,
    number_of_broad_starts=4,
)

**Check the recovered model.** Freeze its geometry and generate Gaussian predictions for new settings. Only after comparing measurements do we open the generating parameters to score the geometry errors.

In [ ]:
checks = calibration.check_full_microscope(
    experiment_directory=experiment,
    microscope=microscope,
)
calibration.show_result(checks=checks)
calibration.show_images(experiment_directory=experiment, fitted_predictions=True);

The error bars describe local uncertainty within this model. Agreement between starts and new-image predictions supports the recovered solution; it does not prove global uniqueness or validate a real microscope's field profile.

Saved stages are reused when their inputs match. Image generation, numerical refinement and fitting logs remain beside the recorded data.